# 🎓 Assistant ENSAM Casablanca — RAG + Claude API
**Démo pas à pas du pipeline** (le projet complet est dans les fichiers `.py` à côté).

```
site ensam-casa.ma ──► scraper.py ──► pages.jsonl ──► build_index.py ──► FAISS + BM25
                                                                             │
question ──► (reformulation, Claude Haiku 5.5) ──► recherche hybride ──► top-8 extraits
                                                                             │
                                       Claude Sonnet 5.5 (streaming) ◄───────┘
                                                 │
                                 réponse + citations [n] + liens sources
```

**Pourquoi Claude au lieu d'un modèle local ?** Qwen 1.5B sur Colab mettait plusieurs dizaines de secondes par réponse et répétait des informations. Avec l'API Claude, la génération tourne sur les serveurs d'Anthropic : réponse en streaming en quelques secondes, meilleure compréhension du français, et aucun GPU nécessaire.

## 0. Installation
Sur Colab : uploadez le dossier `ensam-chatbot` (ou le zip) puis exécutez cette cellule.

In [ ]:
import os, sys
# Colab: dézipper le projet s'il a été uploadé en zip
if os.path.exists("/content/ensam-chatbot.zip") and not os.path.exists("/content/ensam-chatbot"):
    !unzip -q /content/ensam-chatbot.zip -d /content
if os.path.exists("/content/ensam-chatbot"):
    %cd /content/ensam-chatbot
elif os.path.basename(os.getcwd()) == "notebooks":   # exécution locale (Jupyter / VS Code)
    %cd ..
!pip install -q -r requirements.txt
sys.path.insert(0, os.getcwd())

## 1. Clé API Claude
Créez une clé sur https://console.anthropic.com → *API Keys*. Sur Colab, ajoutez-la dans 🔑 *Secrets* sous le nom `ANTHROPIC_API_KEY` (elle n'apparaîtra jamais dans le notebook).

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    if not os.getenv("ANTHROPIC_API_KEY"):
        from getpass import getpass
        os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")
print("Clé chargée :", bool(os.getenv("ANTHROPIC_API_KEY")))

## 2. Scraping du site
Améliorations par rapport à la première version :
- **normalisation des URLs** : `/` et l'URL sans slash ne sont plus scrapées deux fois
- **liste d'exclusion** : login, CGU, archives mensuelles, pages de vérification en triple (`mode=texte`, `mode=image`…)
- **dé-duplication** par empreinte du contenu, et **suppression des menus/pieds de page** répétés
- **lecture des PDF** (règlements, concours, emplois du temps)
- respect de `robots.txt`

> Déjà scrapé ? Sautez cette cellule, `data/pages.jsonl` sera réutilisé.

In [ ]:
import config, scraper
from pathlib import Path

if not config.PAGES_FILE.exists():
    raw = scraper.crawl()
    pages = scraper.clean_pages(raw)
    scraper.save_pages(pages)

import json
pages = [json.loads(l) for l in open(config.PAGES_FILE, encoding="utf-8")]
print(len(pages), "pages,", sum(p["type"] == "pdf" for p in pages), "PDF")
print(pages[1]["title"], "→", pages[1]["url"])
print(pages[1]["text"][:500])

## 3. Découpage en extraits (chunks) + index
Extraits de 250 mots avec 50 mots de chevauchement. Chaque extrait commence par le **titre de sa page** : l'index et Claude savent d'où vient le texte.

In [ ]:
import build_index
chunks = build_index.chunk_pages(pages)
print(len(chunks), "extraits — exemple :\n")
ex = chunks[min(5, len(chunks) - 1)]
print(ex["text"][:400], "\n→", ex["url"])

if not (config.INDEX_DIR / "index.faiss").exists():
    build_index.save(chunks, build_index.build(chunks))

## 4. Recherche hybride
- **Embeddings multilingues + FAISS** : trouvent les passages au *sens* proche (« admission » ≈ « concours d'accès »)
- **BM25** : trouve les *mots exacts*, indispensable pour les sigles (EPEI, PPFI, GI…) que les embeddings comprennent mal
- les deux classements sont fusionnés par **Reciprocal Rank Fusion**

In [ ]:
from rag import Retriever
retriever = Retriever()

for q in ["C'est quoi l'option EPEI ?", "Comment s'inscrire au concours ?"]:
    print("Q:", q)
    for h in retriever.search(q, k=4):
        print(f"   cos={h.dense:.2f}  bm25={h.keyword:5.1f}  mots={h.coverage:.0%}  {h.chunk['title'][:50]}  {h.chunk['url']}")
    print("   pertinent :", retriever.is_relevant(retriever.search(q)), "\n")

## 5. Réponse avec Claude (streaming + citations)
Claude reçoit les extraits numérotés `<source id="n">` et doit citer `[n]` après chaque information. Seules les sources réellement citées sont affichées.

In [ ]:
from rag import SchoolAssistant
from IPython.display import Markdown, display
bot = SchoolAssistant(retriever)

def demo(question, history=None):
    ans = bot.ask(question, history)
    src = "\n".join(f"{s['n']}. [{s['title']}]({s['url']})" for s in ans.sources)
    display(Markdown(f"**Q : {question}**\n\n{ans.text}\n\n{src}\n\n*({ans.latency_s:.1f} s — recherche : « {ans.search_query} »)*"))
    return ans

a1 = demo("Quelles sont les filières d'ingénieur proposées ?")

### Mémoire de conversation
Une question de suivi (« et … ? ») est reformulée par **Claude Haiku 5.5** (rapide et peu cher) avant la recherche.

In [ ]:
history = [{"role": "user", "content": "Quelles sont les filières d'ingénieur proposées ?"},
           {"role": "assistant", "content": a1.text}]
demo("Et quels sont les débouchés de la première ?", history);

### Garde-fous : questions hors sujet
Si aucun extrait n'est pertinent, Claude n'est même pas appelé (0 coût). Sinon, le prompt système lui interdit d'utiliser ses connaissances générales.

In [ ]:
demo("Quelle est la capitale de l'Australie ?")
demo("Écris-moi un poème sur la mer.");

## 6. Évaluation
Jeu de test dans `eval/questions.jsonl` (à compléter avec vos propres questions). Mesure : page attendue retrouvée, page attendue citée, refus des questions hors sujet, latence.

In [ ]:
import evaluate
evaluate.main()
display(Markdown(open("eval/results.md", encoding="utf-8").read()[:3000]))

## 7. Interface web
Lien public temporaire pour la démo. Pour un lien permanent, déployez sur Hugging Face Spaces (voir README).

In [ ]:
import app
app.demo.launch(share=True)